# Example 5: Streaming Pipeline

Live AIS data with sinks and promotion to canonical storage.

This notebook shows the full streaming workflow: connect to a live source,
land data durably via sinks, monitor health, and promote to canonical storage.

## Prerequisites

```bash
pip install neptune-ais[stream,sql,notebooks]
```

**API key required:** You need an [AISStream](https://aisstream.io/) API key to run
the streaming cells. Set your key in the configuration cell below.

## Imports

In [1]:
from neptune_ais.stream import (
    NeptuneStream,
    StreamConfig,
    BackpressurePolicy,
    StreamHealth,
)
from neptune_ais.sinks import ParquetSink, DuckDBSink, promote_landing

## Configure the Stream

A `StreamConfig` defines the source, bounding box, and operational parameters.

| Parameter | Purpose |
|---|---|
| `source` | Adapter name (e.g., `"aisstream"`) |
| `api_key` | Authentication token |
| `bbox` | Geographic bounding box `(west, south, east, north)` |
| `max_queue_size` | Internal buffer limit before backpressure kicks in |
| `backpressure` | `BLOCK` (throttle producer) or `DROP_OLDEST` (prefer freshness) |
| `lag_threshold_s` | Seconds behind real-time before health degrades to `LAGGING` |
| `stale_threshold_s` | Seconds without data before health degrades to `STALE` |

In [ ]:
config = StreamConfig(
    source="aisstream",
    api_key="API_KEY",  # <-- Replace with your key
    bbox=(-74.5, 40.0, -73.5, 41.0),  # New York harbor
    max_queue_size=10_000,
    backpressure=BackpressurePolicy.BLOCK,
    lag_threshold_s=30.0,
    stale_threshold_s=120.0,
)

## Stream to a Parquet Sink

The `ParquetSink` writes incoming messages to partitioned Parquet files in a
landing directory. `NeptuneStream` manages the WebSocket connection, message
parsing, and health monitoring.

**Health states:** `HEALTHY` → `LAGGING` → `STALE` → `DISCONNECTED`

> **Note:** In Jupyter, you can use top-level `await` directly — no need for `asyncio.run()`.

In [3]:
sink = ParquetSink("/tmp/neptune_landing", source="aisstream")

async with NeptuneStream(config=config) as stream:
    # Monitor health during streaming
    print(f"Health: {stream.health.value}")

    # stream.run() starts the WebSocket connection and sink concurrently
    await stream.run(sink, max_messages=1000)

    # Check final health
    snap = stream.health_snapshot()
    print(f"Health: {snap['health']}")
    print(f"Received: {snap['messages_received']}")
    print(f"Delivered: {snap['messages_delivered']}")
    print(f"Dedup rate: {snap['dedup_rate']:.1%}")

Health: healthy
Health: disconnected
Received: 1000
Delivered: 1000
Dedup rate: 0.0%


## Promote Landed Data to Canonical Storage

Once data has landed in Parquet files, `promote_landing()` normalizes it,
runs QC, and writes it to the canonical store — the same format used by
the archival path.

In [4]:
results = promote_landing(
    landing_dir="/tmp/neptune_landing",
    store_root="/tmp/neptune_demo",
    source="aisstream",
    cleanup=True,
)
for r in results:
    print(f"  {r.date}: {r.record_count:,} rows promoted")

  2026-03-20: 1,020 rows promoted


## Alternative: DuckDB Sink

The `DuckDBSink` writes directly to a DuckDB database, giving you
immediate SQL access to landed data without a separate promotion step.

In [5]:
sink = DuckDBSink(db_path="/tmp/ais_live.duckdb", table_name="positions")

async with NeptuneStream(config=config) as stream:
    await stream.run(sink, max_messages=500)

## Next Steps

- **[06 — External Plugin](06_external_plugin.py)**: Build a custom source adapter (Python script — copy and modify)
- Combine streaming data with archival data using [Multi-Source Fusion](03_multi_source_fusion.ipynb)